In [1]:
# importing libraries

from scipy.io import loadmat
import os
import numpy as np
import pandas as pd




In [2]:
# loading a single mat file

data = loadmat('../../DataSets/Complete_700_DataPoints/Simulation_DC_FHDS_extended_SF_-2.50_SOC_H0.51_M0.54_L0.76_w1_0.57_w2_0.96_w3_0.75_wPL_0.000062_Sim292.mat')

In [3]:
print(data.keys())

# sample data keys
# dict_keys(['__header__', '__version__', '__globals__', 'D1', 'DC', 'H', 'I_b', 'I_cd', 'I_cells', 'J', 'L', 'M', 'PLoss', 'P_IO', 'S_factor', 'range', 'sim_idx', 'soc_M', 't_sim', 'v_t', 'w1', 'w2', 'w3', 'w_PL'])


dict_keys(['__header__', '__version__', '__globals__', 'D1', 'DC', 'H', 'I_b', 'I_cd', 'I_cells', 'J', 'L', 'M', 'PLoss', 'P_IO', 'S_factor', 'range', 'sim_idx', 'soc_M', 't_sim', 'v_t', 'w1', 'w2', 'w3', 'w_PL'])


In [4]:
# printing w1, w2, w3, wPL and range last value

print(data['w1'])
print(data['w2'])
print(data['w3'])
print(data['w_PL'])
print(data['range'][-1])


[[0.57]]
[[0.96]]
[[0.75]]
[[6.18091239e-05]]
[154.49666667]


In [5]:
# Getting the list of all the files in the directory
files = os.listdir('../../DataSets/Complete_700_DataPoints/')
print(f"Total files: {len(files)}")

Total files: 703


In [6]:
print(len(files))

703


In [7]:
# Initialize an empty DataFrame
df = pd.DataFrame(columns=['w1', 'w2', 'w3', 'wPL', 'range'])

In [8]:
# List to collect rows for the DataFrame
rows = []

# Iterating over all the files and adding the values to the rows list
for file in files:
    file_path = os.path.join('../../DataSets/Complete_700_DataPoints/', file)
    try:
        # Check if the file size is greater than zero
        if os.path.getsize(file_path) > 0:
            data = loadmat(file_path)
            # Check if all required keys are present in the file
            if 'w1' in data and 'w2' in data and 'w3' in data and 'w_PL' in data and 'range' in data:
                # Append a new row to the list
                rows.append({
                    'w1': data['w1'][0][0], 
                    'w2': data['w2'][0][0], 
                    'w3': data['w3'][0][0], 
                    'wPL': data['w_PL'][0][0], 
                    'range': data['range'][-1][0]
                })
            else:
                print(f"Skipping file with missing data: {file}")
        else:
            print(f"Skipping empty file: {file}")
    except Exception as e:
        print(f"Error reading {file}: {e}")

Skipping empty file: Icon


In [9]:
# Creating the DataFrame from the collected rows
df = pd.DataFrame(rows)

In [10]:
# Display the final DataFrame shape and check for duplicates
print(f"DataFrame shape: {df.shape}")
print(f"Number of unique rows: {df.drop_duplicates().shape[0]}")

DataFrame shape: (702, 5)
Number of unique rows: 700


In [11]:
# printing duplicate rows

print(df[df.duplicated()])

       w1    w2    w3       wPL       range
387  0.83  0.39  0.61  0.000044  167.661833
536  0.18  0.16  0.67  0.000082  159.237833


In [12]:
# Formatting the wPL column to %.6f
df['wPL'] = df['wPL'].map(lambda x: f"{x:.6f}")

# Saving the DataFrame to a CSV file
df.to_csv('output_file.csv', index=False)

# Display the resulting DataFrame
print(df)

       w1    w2    w3       wPL       range
0    0.19  0.63  0.94  0.000029  168.577083
1    0.43  0.69  0.95  0.000073  193.161944
2    0.62  0.12  0.12  0.000062  125.559007
3    0.53  0.65  0.41  0.000032   98.348028
4    0.41  0.44  0.18  0.000074  134.261982
..    ...   ...   ...       ...         ...
697  0.48  0.68  0.21  0.000028  147.791200
698  0.39  0.76  0.62  0.000048  168.577083
699  0.92  0.35  0.22  0.000075  168.577083
700  0.95  0.54  0.54  0.000073  205.729167
701  0.22  0.13  0.31  0.000041  167.661833

[702 rows x 5 columns]


In [13]:
from scipy.io import savemat


In [14]:
# Ensure the DataFrame has the expected columns
required_columns = ['w1', 'w2', 'w3', 'wPL', 'range']
if not all(col in df.columns for col in required_columns):
    raise ValueError("DataFrame is missing one or more required columns.")

In [15]:
# Prepare the matrices for saving
data_to_save = {
    'weights': df[['w1', 'w2', 'w3', 'wPL']].to_numpy(),  # N x 4 matrix
    'range': df[['range']].to_numpy()                  # N x 1 matrix
}

In [16]:
# Save the data to a .mat file
savemat('test_file.mat', data_to_save)